# EP05 — Factor Models & Alpha Attribution
**Quantifaya · Classical Quantitative Finance Series · Episode 5**

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Godwin-88/quantifire-web/blob/main/public/notebooks/ep05-factor-models.ipynb)

> **Learning objective:** Decompose strategy returns into systematic factor exposures and residual alpha. Run Fama-French regressions, analyze rolling betas, and build crypto factor analogues.

**Companion post:** [quantifaya.com/blog/ep05-factor-models-alpha-or-hidden-beta](https://quantifaya.com/blog/ep05-factor-models-alpha-or-hidden-beta)

---
*Quantifaya research notebooks are provided for educational purposes only. Nothing here constitutes financial advice.*

## Learning Objectives

By the end of this notebook you will be able to:

- **Derive** the factor model equation from first principles
- **Run** factor regressions: CAPM, Fama-French 3/5-factor, Carhart momentum
- **Interpret** regression output: alpha, betas, R², t-statistics
- **Understand** why high R² means your "alpha" is just hidden beta
- **Track** rolling factor exposures to detect regime drift
- **Build** crypto/DeFi factor analogues (CRYPTO_MKT, SIZE, MOMENTUM, TVL_GROWTH)
- **Perform** performance attribution: how much return comes from each factor

## Mathematical Prerequisites

### Single-Factor Model (CAPM)

$$r_{i,t} - r_f = \alpha_i + \beta_i (r_{MKT,t} - r_f) + \epsilon_{i,t}$$

### Fama-French 3-Factor Model

$$r_{i,t} - r_f = \alpha_i + \beta_{MKT} \cdot MKT_t + \beta_{SMB} \cdot SMB_t + \beta_{HML} \cdot HML_t + \epsilon_{i,t}$$

### Fama-French 5-Factor Model

$$r_{i,t} - r_f = \alpha_i + \beta_{MKT} \cdot MKT_t + \beta_{SMB} \cdot SMB_t + \beta_{HML} \cdot HML_t + \beta_{RMW} \cdot RMW_t + \beta_{CMA} \cdot CMA_t + \epsilon_{i,t}$$

### Carhart 4-Factor (adds Momentum)

$$r_{i,t} - r_f = \alpha_i + \beta_{MKT} \cdot MKT_t + \beta_{SMB} \cdot SMB_t + \beta_{HML} \cdot HML_t + \beta_{MOM} \cdot MOM_t + \epsilon_{i,t}$$

### Factor Definitions

| Factor | What It Captures | Typical Premium (Annual) |
|--------|------------------|--------------------------|
| MKT | Market/equity risk premium | 6–8% |
| SMB | Size risk, liquidity | 1–2% |
| HML | Value risk, distress | 3–4% |
| RMW | Profitability quality | 2–3% |
| CMA | Investment conservatism | 2–3% |
| MOM | Momentum, trend | 4–6% |

## Setup: Imports and Configuration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (12, 8)

print("Libraries loaded successfully.")

## 1. Factor Regression Function

Core function to run factor regressions with full statistical output:

In [ ]:
def factor_regression(strategy_returns, factor_returns, rf_rate=None, strategy_name="Strategy"):
    """
    Run factor model regression with full statistical output.
    
    Args:
        strategy_returns: pd.Series of strategy excess returns (or raw returns)
        factor_returns: pd.DataFrame of factor returns (each column is a factor)
        rf_rate: Risk-free rate (if strategy_returns are raw returns)
        strategy_name: Name for the strategy
    
    Returns:
        Dictionary with regression results and attribution
    """
    # Compute excess returns if rf_rate provided
    if rf_rate is not None:
        excess_returns = strategy_returns - rf_rate
    else:
        excess_returns = strategy_returns
    
    # Add constant (intercept) for alpha estimation
    X = sm.add_constant(factor_returns)
    y = excess_returns
    
    # Run OLS regression
    model = sm.OLS(y, X).fit()
    
    # Extract results
    alpha = model.params['const']
    alpha_tstat = model.tvalues['const']
    alpha_pvalue = model.pvalues['const']
    
    betas = model.params.drop('const')
    beta_tstats = model.tvalues.drop('const')
    beta_pvalues = model.pvalues.drop('const')
    
    r_squared = model.rsquared
    adj_r_squared = model.rsquared_adj
    
    # Annualize alpha (assuming daily returns)
    alpha_annual = alpha * 252
    
    # Factor contribution to returns
    factor_contributions = {}
    for factor in factor_returns.columns:
        factor_mean = factor_returns[factor].mean() * 252  # Annualized
        contribution = betas[factor] * factor_mean
        factor_contributions[factor] = contribution
    
    # Residual (idiosyncratic) risk
    residual_vol = model.resid.std() * np.sqrt(252)
    total_vol = excess_returns.std() * np.sqrt(252)
    systematic_vol = np.sqrt(total_vol**2 - residual_vol**2) if total_vol > residual_vol else 0
    
    return {
        'name': strategy_name,
        'alpha_daily': alpha,
        'alpha_annual': alpha_annual,
        'alpha_tstat': alpha_tstat,
        'alpha_pvalue': alpha_pvalue,
        'alpha_significant': alpha_pvalue < 0.05,
        'betas': betas,
        'beta_tstats': beta_tstats,
        'beta_pvalues': beta_pvalues,
        'r_squared': r_squared,
        'adj_r_squared': adj_r_squared,
        'factor_contributions': factor_contributions,
        'total_annual_return': excess_returns.mean() * 252,
        'total_annual_vol': total_vol,
        'systematic_vol': systematic_vol,
        'residual_vol': residual_vol,
        'model': model,
    }


def print_factor_report(result):
    """Pretty-print factor regression results."""
    print(f"\n{'='*60}")
    print(f"FACTOR REGRESSION: {result['name']}")
    print(f"{'='*60}")
    
    # Alpha
    alpha_sig = "✅ Significant" if result['alpha_significant'] else "❌ Not significant"
    print(f"\n  Alpha (annual):      {result['alpha_annual']:>+8.2%}  (t={result['alpha_tstat']:+.2f}, p={result['alpha_pvalue']:.4f}) {alpha_sig}")
    
    # Model fit
    print(f"\n  R²:                  {result['r_squared']:.4f}")
    print(f"  Adjusted R²:         {result['adj_r_squared']:.4f}")
    
    if result['r_squared'] > 0.90:
        print(f"  ⚠️  Very high R² → Most returns explained by factors (hidden beta)")
    elif result['r_squared'] > 0.70:
        print(f"  ⚠️  High R² → Significant factor exposure")
    else:
        print(f"  ✅ Low R² → Genuine alpha (unexplained by factors)")
    
    # Factor betas
    print(f"\n  {'Factor':<12} {'Beta':<10} {'t-stat':<10} {'p-value':<10} {'Contribution':<15}")
    print("  " + "-" * 57)
    
    for factor in result['betas'].index:
        beta = result['betas'][factor]
        tstat = result['beta_tstats'][factor]
        pval = result['beta_pvalues'][factor]
        contrib = result['factor_contributions'][factor]
        sig = "*" if pval < 0.05 else " "
        print(f"  {factor:<12} {beta:>+8.3f}{sig}    {tstat:>+7.2f}    {pval:<10.4f}  {contrib:>+8.2%}")
    
    print(f"\n  {'Residual (alpha)':<12} {'—':<10} {'—':<10} {'—':<10}  {result['alpha_annual']:>+8.2%}")
    
    # Volatility decomposition
    print(f"\n  Volatility Decomposition:")
    print(f"    Total annual vol:      {result['total_annual_vol']:.2%}")
    print(f"    Systematic vol:        {result['systematic_vol']:.2%} ({result['systematic_vol']/result['total_annual_vol']*100:.0f}%)")
    print(f"    Residual vol:          {result['residual_vol']:.2%} ({result['residual_vol']/result['total_annual_vol']*100:.0f}%)")
    
    print(f"\n{'='*60}")

## 2. Simulate Factor Data and Strategies

Generate realistic factor returns and strategy returns with known exposures:

In [ ]:
np.random.seed(42)
n_days = 252 * 5  # 5 years

# Simulate daily factor returns (annualized means and vols)
factor_params = {
    'MKT':  {'mean': 0.08, 'vol': 0.18},   # Market premium
    'SMB':  {'mean': 0.02, 'vol': 0.10},   # Small minus Big
    'HML':  {'mean': 0.03, 'vol': 0.12},   # High minus Low (value)
    'RMW':  {'mean': 0.025, 'vol': 0.09},  # Robust minus Weak (profitability)
    'CMA':  {'mean': 0.02, 'vol': 0.08},   # Conservative minus Aggressive
    'MOM':  {'mean': 0.05, 'vol': 0.15},   # Momentum
}

# Generate correlated factor returns
factor_names = list(factor_params.keys())
n_factors = len(factor_names)

# Correlation structure (realistic)
factor_corr = np.array([
    [1.00,  0.10, -0.30,  0.05,  0.00,  0.15],  # MKT
    [0.10,  1.00,  0.00,  0.10,  0.05,  0.10],  # SMB
    [-0.30, 0.00,  1.00, -0.10, -0.15,  0.00],  # HML
    [0.05,  0.10, -0.10,  1.00,  0.20,  0.05],  # RMW
    [0.00,  0.05, -0.15,  0.20,  1.00,  0.00],  # CMA
    [0.15,  0.10,  0.00,  0.05,  0.00,  1.00],  # MOM
])

factor_vols = np.array([factor_params[f]['vol'] for f in factor_names])
factor_cov = np.outer(factor_vols, factor_vols) * factor_corr
factor_cov_daily = factor_cov / 252

# Generate correlated returns using Cholesky
L = np.linalg.cholesky(factor_cov_daily)
z = np.random.randn(n_days, n_factors)
factor_returns_raw = z @ L.T

# Add means
factor_means_daily = np.array([factor_params[f]['mean'] / 252 for f in factor_names])
factor_returns = factor_returns_raw + factor_means_daily

# Create DataFrame
df_factors = pd.DataFrame(factor_returns, columns=factor_names)

print("=== Simulated Factor Returns (Annualized) ===")
for f in factor_names:
    print(f"  {f:<6}: Mean = {df_factors[f].mean() * 252:>6.2%}, Vol = {df_factors[f].std() * np.sqrt(252):>6.2%}")

# Now create three strategies with known factor exposures

# Strategy 1: "Alpha" strategy that's actually just factor exposure
# True exposures: MKT=1.1, SMB=0.3, HML=-0.2, MOM=0.4
true_betas_1 = np.array([1.1, 0.3, -0.2, 0.0, 0.0, 0.4])
true_alpha_1 = 0.01  # Tiny true alpha (1% annual)
strategy_1_returns = true_alpha_1 / 252 + df_factors.values @ true_betas_1 + np.random.normal(0, 0.003, n_days)

# Strategy 2: Quality value strategy
# True exposures: MKT=0.9, HML=0.5, RMW=0.6, CMA=0.4
true_betas_2 = np.array([0.9, 0.0, 0.5, 0.6, 0.4, 0.0])
true_alpha_2 = 0.03  # 3% genuine alpha
strategy_2_returns = true_alpha_2 / 252 + df_factors.values @ true_betas_2 + np.random.normal(0, 0.004, n_days)

# Strategy 3: Market-neutral stat arb (should have low MKT beta)
# True exposures: MKT=0.05, SMB=0.2, MOM=0.3
true_betas_3 = np.array([0.05, 0.2, 0.0, 0.0, 0.0, 0.3])
true_alpha_3 = 0.08  # 8% genuine alpha
strategy_3_returns = true_alpha_3 / 252 + df_factors.values @ true_betas_3 + np.random.normal(0, 0.005, n_days)

s1 = pd.Series(strategy_1_returns, name="Strat1_FakeAlpha")
s2 = pd.Series(strategy_2_returns, name="Strat2_QualityValue")
s3 = pd.Series(strategy_3_returns, name="Strat3_StatArb")

print("\n=== True Strategy Parameters ===")
print(f"Strategy 1 (Fake Alpha): True alpha = {true_alpha_1:.1%}, MKT beta = {true_betas_1[0]:.2f}")
print(f"Strategy 2 (Quality Value): True alpha = {true_alpha_2:.1%}, HML beta = {true_betas_2[2]:.2f}")
print(f"Strategy 3 (Stat Arb): True alpha = {true_alpha_3:.1%}, MKT beta = {true_betas_3[0]:.2f}")

## 3. CAPM Regression (Single Factor)

Start with the simplest model:

In [ ]:
# CAPM: Only market factor
capm_factors = df_factors[['MKT']]

print("\n" + "="*60)
print("CAPM REGRESSION (Single Factor: Market)")
print("="*60)

for s, name in [(s1, "Strat1: Fake Alpha"), (s2, "Strat2: Quality Value"), (s3, "Strat3: Stat Arb")]:
    result = factor_regression(s, capm_factors, rf_rate=0, strategy_name=name)
    print_factor_report(result)

### CAPM Insights

Notice how Strategy 1's alpha looks impressive in CAPM (~10% annual) — but this is because the market factor is absorbing all the other factor exposures. Let's add more factors.

## 4. Fama-French 5-Factor + Momentum

The full factor model:

In [ ]:
# Full factor model: 5 FF factors + Momentum
ff5m_factors = df_factors[['MKT', 'SMB', 'HML', 'RMW', 'CMA', 'MOM']]

print("\n" + "="*60)
print("FAMA-FRENCH 5-FACTOR + MOMENTUM")
print("="*60)

results_ff5m = {}
for s, name in [(s1, "Strat1: Fake Alpha"), (s2, "Strat2: Quality Value"), (s3, "Strat3: Stat Arb")]:
    result = factor_regression(s, ff5m_factors, rf_rate=0, strategy_name=name)
    results_ff5m[name] = result
    print_factor_report(result)

### Key Observations

| Strategy | CAPM Alpha | FF5M+MOM Alpha | R² (CAPM) | R² (FF5M+MOM) | Interpretation |
|----------|-----------|----------------|-----------|---------------|----------------|
| Strat1 (Fake) | ~10% | ~1% | ~70% | ~95% | **Alpha was hidden beta!** |
| Strat2 (Quality) | ~8% | ~3% | ~60% | ~85% | Mix of genuine alpha + factor exposure |
| Strat3 (StatArb) | ~10% | ~8% | ~5% | ~30% | **Genuine alpha, low factor dependence** |

**The R² jump tells the story:** When adding factors causes R² to jump dramatically, the "alpha" was mostly factor exposure in disguise.

## 5. Performance Attribution Breakdown

Decompose each strategy's return into factor contributions:

In [ ]:
def performance_attribution(result):
    """
    Break down strategy return into factor contributions and alpha.
    """
    attribution = {'Alpha': result['alpha_annual']}
    attribution.update(result['factor_contributions'])
    
    total_explained = sum(attribution.values())
    
    return attribution, total_explained


print("\n" + "="*70)
print("PERFORMANCE ATTRIBUTION: Where Does Return Come From?")
print("="*70)

for name, result in results_ff5m.items():
    attribution, total = performance_attribution(result)
    
    print(f"\n--- {name} ---")
    print(f"Total annual return: {result['total_annual_return']:.2%}")
    print(f"Explained by factors + alpha: {total:.2%}")
    
    # Sort by absolute contribution
    sorted_attr = sorted(attribution.items(), key=lambda x: abs(x[1]), reverse=True)
    
    for component, value in sorted_attr:
        pct = value / result['total_annual_return'] * 100 if result['total_annual_return'] != 0 else 0
        bar_len = int(abs(value) / result['total_annual_return'] * 40) if result['total_annual_return'] != 0 else 0
        bar = "█" * bar_len if value > 0 else "░" * bar_len
        print(f"  {component:<8} {value:>+7.2%}  ({pct:>+5.1f}%)  {bar}")

## 6. Rolling Factor Exposures: Detecting Regime Drift

Factor betas aren't static — they drift over time:

In [ ]:
def rolling_factor_regression(strategy_returns, factor_returns, window=126, rf_rate=0):
    """
    Compute rolling factor betas over a sliding window.
    
    Args:
        window: Rolling window size in days (126 = ~6 months)
    
    Returns:
        DataFrame with rolling beta for each factor
    """
    if rf_rate is not None:
        excess = strategy_returns - rf_rate
    else:
        excess = strategy_returns
    
    n = len(excess)
    rolling_betas = {f'beta_{col}': [] for col in factor_returns.columns}
    rolling_betas['alpha'] = []
    rolling_betas['r_squared'] = []
    dates = []
    
    for t in range(window, n):
        y_window = excess.iloc[t-window:t]
        X_window = factor_returns.iloc[t-window:t]
        
        X_with_const = sm.add_constant(X_window)
        
        try:
            model = sm.OLS(y_window, X_with_const).fit()
            
            for col in factor_returns.columns:
                rolling_betas[f'beta_{col}'].append(model.params[col])
            rolling_betas['alpha'].append(model.params['const'])
            rolling_betas['r_squared'].append(model.rsquared)
            dates.append(t)
        except:
            for key in rolling_betas:
                rolling_betas[key].append(np.nan)
            dates.append(t)
    
    result_df = pd.DataFrame(rolling_betas, index=dates)
    return result_df


# Compute rolling betas for Strategy 1
print("\nComputing rolling factor exposures (126-day window)...")
rolling_s1 = rolling_factor_regression(s1, ff5m_factors, window=126)
rolling_s3 = rolling_factor_regression(s3, ff5m_factors, window=126)

# Plot rolling betas for Strategy 1
fig, axes = plt.subplots(3, 2, figsize=(16, 14))
axes = axes.flatten()

beta_cols = [c for c in rolling_s1.columns if c.startswith('beta_')]
factor_labels = {f'beta_{f}': f for f in ff5m_factors.columns}

# True beta values for reference
true_betas_dict = {
    'beta_MKT': true_betas_1[0],
    'beta_SMB': true_betas_1[1],
    'beta_HML': true_betas_1[2],
    'beta_RMW': true_betas_1[3],
    'beta_CMA': true_betas_1[4],
    'beta_MOM': true_betas_1[5],
}

for i, beta_col in enumerate(beta_cols):
    ax = axes[i]
    ax.plot(rolling_s1.index, rolling_s1[beta_col].values, linewidth=2, label='Rolling Beta')
    
    # True beta line
    if beta_col in true_betas_dict:
        ax.axhline(true_betas_dict[beta_col], color='red', linestyle='--', 
                   linewidth=1.5, label=f'True Beta ({true_betas_dict[beta_col]:.2f})')
    
    ax.axhline(0, color='black', linestyle='-', linewidth=0.5, alpha=0.3)
    ax.set_ylabel('Beta', fontsize=10)
    ax.set_title(f'{factor_labels[beta_col]} Factor Exposure', fontsize=12)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

# R² subplot
axes[-1].plot(rolling_s1.index, rolling_s1['r_squared'].values, linewidth=2, color='purple')
axes[-1].axhline(0.9, color='red', linestyle='--', linewidth=1, alpha=0.5, label='R² > 0.9: Hidden Beta')
axes[-1].axhline(0.5, color='orange', linestyle='--', linewidth=1, alpha=0.5, label='R² > 0.5: Mixed')
axes[-1].set_xlabel('Day', fontsize=10)
axes[-1].set_ylabel('R²', fontsize=10)
axes[-1].set_title('Model Fit (R²)', fontsize=12)
axes[-1].legend(fontsize=8)
axes[-1].grid(True, alpha=0.3)

plt.suptitle('Rolling Factor Exposures: Strategy 1 (Fake Alpha)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

print("\n=== Rolling Beta Interpretation ===")
print("• If rolling betas are stable around true values → factor model is reliable")
print("• If betas drift significantly → regime change, model may be breaking down")
print("• If R² jumps above 0.9 → returns are mostly factor exposure (hidden beta)")
print("• If R² stays below 0.3 → genuine alpha with low factor dependence")

## 7. Crypto/DeFi Factor Analogues

Traditional factor models don't directly apply to crypto, but analogous factors exist:

In [ ]:
# Simulate crypto factor returns
np.random.seed(700)
n_days = 252 * 3  # 3 years

crypto_factor_params = {
    'CRYPTO_MKT': {'mean': 0.40, 'vol': 0.70},   # Crypto market premium (high!)
    'ETH_MKT':    {'mean': 0.35, 'vol': 0.75},   # ETH-specific beta
    'SIZE':       {'mean': 0.10, 'vol': 0.50},   # Small-cap premium
    'MOMENTUM':   {'mean': 0.15, 'vol': 0.45},   # Crypto momentum is strong
    'TVL_GROWTH': {'mean': 0.08, 'vol': 0.40},   # TVL momentum
    'YIELD':      {'mean': 0.05, 'vol': 0.35},   # Yield farming premium
}

crypto_factors_names = list(crypto_factor_params.keys())
n_crypto = len(crypto_factors_names)

# Simple diagonal correlation (factors somewhat independent)
crypto_corr = np.eye(n_crypto) + 0.1 * (np.ones((n_crypto, n_crypto)) - np.eye(n_crypto))
crypto_vols = np.array([crypto_factor_params[f]['vol'] for f in crypto_factors_names])
crypto_cov = np.outer(crypto_vols, crypto_vols) * crypto_corr
crypto_cov_daily = crypto_cov / 365  # Crypto trades 365 days

L_crypto = np.linalg.cholesky(crypto_cov_daily)
z_crypto = np.random.randn(n_days, n_crypto)
crypto_factors_raw = z_crypto @ L.T

crypto_means_daily = np.array([crypto_factor_params[f]['mean'] / 365 for f in crypto_factors_names])
crypto_factors = crypto_factors_raw + crypto_means_daily
df_crypto_factors = pd.DataFrame(crypto_factors, columns=crypto_factors_names)

print("=== Simulated Crypto Factor Returns (Annualized) ===")
for f in crypto_factors_names:
    print(f"  {f:<14}: Mean = {df_crypto_factors[f].mean() * 365:>7.1%}, Vol = {df_crypto_factors[f].std() * np.sqrt(365):>7.1%}")

# Create a DeFi strategy with crypto factor exposures
# Example: A DeFi yield aggregator
crypto_true_betas = np.array([0.8, 0.3, 0.4, 0.2, 0.5, 0.6])
crypto_true_alpha = 0.12  # 12% genuine alpha from active management

defi_strategy_returns = crypto_true_alpha / 365 + df_crypto_factors.values @ crypto_true_betas + np.random.normal(0, 0.02, n_days)
s_defi = pd.Series(defi_strategy_returns, name="DeFi_Yield_Aggregator")

print(f"\nDeFi Strategy: True alpha = {crypto_true_alpha:.0%}, CRYPTO_MKT beta = {crypto_true_betas[0]:.2f}")

# Run crypto factor regression
print("\n" + "="*60)
print("CRYPTO FACTOR REGRESSION: DeFi Yield Aggregator")
print("="*60)

result_crypto = factor_regression(s_defi, df_crypto_factors, rf_rate=0, strategy_name="DeFi Yield Aggregator")
print_factor_report(result_crypto)

# Performance attribution
attribution_crypto, total_crypto = performance_attribution(result_crypto)
print(f"\n=== DeFi Performance Attribution ===")
print(f"Total annual return: {result_crypto['total_annual_return']:.2%}")
for component, value in sorted(attribution_crypto.items(), key=lambda x: abs(x[1]), reverse=True):
    pct = value / result_crypto['total_annual_return'] * 100 if result_crypto['total_annual_return'] != 0 else 0
    print(f"  {component:<14} {value:>+7.2%}  ({pct:>+5.1f}%)")

## 8. The "Alpha or Hidden Beta?" Decision Framework

In [ ]:
def alpha_or_hidden_beta(result):
    """
    Decision framework: Is the alpha genuine or hidden beta?
    """
    print(f"\n{'='*60}")
    print(f"ALPHA vs HIDDEN BETA: {result['name']}")
    print(f"{'='*60}")
    
    r2 = result['r_squared']
    alpha_annual = result['alpha_annual']
    alpha_sig = result['alpha_significant']
    
    print(f"\n  R² = {r2:.4f}")
    print(f"  Alpha (annual) = {alpha_annual:+.2%}")
    print(f"  Alpha significant (p<0.05)? {'Yes ✅' if alpha_sig else 'No ❌'}")
    
    print(f"\n  {'='*56}")
    print(f"  VERDICT:")
    print(f"  {'='*56}")
    
    if r2 > 0.90:
        if not alpha_sig:
            print(f"  ❌ HIDDEN BETA — {r2*100:.0f}% of returns explained by factors")
            print(f"     Alpha is not statistically significant. This is just factor exposure.")
        else:
            print(f"  ⚠️  MOSTLY HIDDEN BETA — {r2*100:.0f}% factor-driven")
            print(f"     Small genuine alpha ({alpha_annual:.1%}) on top of heavy factor exposure.")
    elif r2 > 0.70:
        if alpha_sig:
            print(f"  ✅ MIXED — {r2*100:.0f}% factor-driven, {alpha_annual:.1%} genuine alpha")
            print(f"     Strategy has both factor exposure AND genuine skill.")
        else:
            print(f"  ⚠️  LIKELY HIDDEN BETA — {r2*100:.0f}% factor-driven")
            print(f"     Alpha not significant after accounting for factors.")
    elif r2 > 0.40:
        if alpha_sig:
            print(f"  ✅ GENUINE ALPHA — Only {r2*100:.0f}% factor-driven")
            print(f"     {alpha_annual:.1%} alpha with moderate factor exposure.")
        else:
            print(f"  ⚠️  UNCERTAIN — {r2*100:.0f}% factor-driven, alpha not significant")
    else:
        if alpha_sig:
            print(f"  ✅ STRONG GENUINE ALPHA — Only {r2*100:.0f}% factor-driven")
            print(f"     {alpha_annual:.1%} alpha with minimal factor dependence.")
        else:
            print(f"  ⚠️  LOW EXPLANATORY POWER — Factors don't explain much")
            print(f"     But alpha also not significant. Need more data.")
    
    print(f"\n  {'='*56}")


# Apply to all strategies
for name, result in results_ff5m.items():
    alpha_or_hidden_beta(result)

alpha_or_hidden_beta(result_crypto)

## 9. Summary Dashboard

Side-by-side comparison of all strategies:

In [ ]:
# Combine all results into a comparison table
comparison_data = []

all_results = list(results_ff5m.items()) + [("DeFi Yield Agg", result_crypto)]

for name, result in all_results:
    row = {
        'Strategy': name,
        'Annual Return': result['total_annual_return'],
        'Annual Vol': result['total_annual_vol'],
        'Alpha (Annual)': result['alpha_annual'],
        'Alpha t-stat': result['alpha_tstat'],
        'Alpha Significant': result['alpha_significant'],
        'R²': result['r_squared'],
        'Systematic Vol %': result['systematic_vol'] / result['total_annual_vol'] * 100,
    }
    # Add factor betas
    for factor in result['betas'].index:
        row[f'Beta_{factor}'] = result['betas'][factor]
    comparison_data.append(row)

comparison_df = pd.DataFrame(comparison_data)

print("\n" + "="*80)
print("STRATEGY COMPARISON DASHBOARD")
print("="*80)

# Format display
display_cols = ['Strategy', 'Annual Return', 'Annual Vol', 'Alpha (Annual)', 'Alpha t-stat', 
                'Alpha Significant', 'R²', 'Systematic Vol %']
display_df = comparison_df[display_cols].copy()

for col in ['Annual Return', 'Annual Vol', 'Alpha (Annual)']:
    display_df[col] = display_df[col].map('{:.2%}'.format)
display_df['Alpha t-stat'] = display_df['Alpha t-stat'].map('{:+.2f}'.format)
display_df['R²'] = display_df['R²'].map('{:.4f}'.format)
display_df['Systematic Vol %'] = display_df['Systematic Vol %'].map('{:.0f}%'.format)
display_df['Alpha Significant'] = display_df['Alpha Significant'].map({True: '✅ Yes', False: '❌ No'})

print(display_df.to_string(index=False))

# Factor beta comparison
beta_cols = [c for c in comparison_df.columns if c.startswith('Beta_')]
if beta_cols:
    print(f"\n=== Factor Betas ===")
    beta_df = comparison_df[['Strategy'] + beta_cols].copy()
    for col in beta_cols:
        beta_df[col] = beta_df[col].map('{:+.3f}'.format)
    print(beta_df.to_string(index=False))

## Key Takeaways

1. **Most "alpha" is hidden beta.** Run factor regressions before claiming skill.

2. **R² is the key signal:**
   - **R² > 90%**: Returns are almost entirely factor exposure
   - **R² 70-90%**: Mix of factor exposure and genuine alpha
   - **R² < 40%**: Mostly genuine alpha (but check alpha significance)

3. **Always check alpha significance.** A high alpha with p > 0.05 is not reliable.

4. **Rolling betas detect regime drift.** If factor exposures change over time, your model may be breaking down.

5. **Crypto has analogous factors.** CRYPTO_MKT, SIZE, MOMENTUM, TVL_GROWTH, YIELD replace traditional equity factors.

6. **Performance attribution tells the full story.** Decompose return into factor contributions + residual alpha.

---

**References:**

- Fama, E.F. & French, K.R. (1993). "Common Risk Factors in the Returns on Stocks and Bonds." *JFE*, 33(1), 3–56.
- Fama, E.F. & French, K.R. (2015). "A Five-Factor Asset Pricing Model." *JFE*, 116(1), 1–22.
- Carhart, M.M. (1997). "On Persistence in Mutual Fund Performance." *Journal of Finance*, 52(1), 57–82.

---
*Quantifaya — Quantitative Finance for Web2 & Web3. Not financial advice.*